# 1. VLM Prompt Engineering — Caption / OCR / Reasoning / Structured Output

Qwen3-VL(Ollama)을 호출하며 프롬프트 작성 방식에 따라 결과가 어떻게
달라지는지 직접 비교합니다.

## 사전 준비
```bash
uv sync
ollama pull qwen3-vl:4b
```
(최초 1회) `data/generate_sample_data.py`로 실습 이미지를 생성해 두었는지 확인:
```bash
uv run python data/generate_sample_data.py
```

In [ ]:
from vlm_utils import ask, ask_json, show

PRODUCT = "data/product.png"
RECEIPT = "data/receipt.png"
BAR_CHART = "data/bar_chart.png"

## Caption Prompt
같은 이미지라도 "범위 / 관점 / 용도"를 지정하지 않으면 VLM은 스스로
분량과 톤을 정해버립니다. 나쁜 예 / 좋은 예를 나란히 비교해봅니다.

In [ ]:
show(PRODUCT)

In [ ]:
bad = ask(PRODUCT, "이 이미지를 설명해줘.")
print("[나쁜 예]\n", bad)

In [ ]:
good = ask(
    PRODUCT,
    "이 상품 이미지를 온라인 쇼핑몰 상세페이지용 alt 텍스트로 쓸 수 있게, "
    "색상·소재·형태를 포함해 한국어 한 문장으로 설명해줘.",
)
print("[좋은 예]\n", good)

## OCR Prompt
"읽어줘" 대신 **"그대로 전사(transcribe)해줘"**라고 요청하면 임의 요약/의역이
줄어듭니다.

In [ ]:
show(RECEIPT)

In [ ]:
bad_ocr = ask(RECEIPT, "이 이미지에 뭐라고 써있어?")
print("[나쁜 예]\n", bad_ocr)

In [ ]:
good_ocr = ask(
    RECEIPT,
    "이미지 속 모든 텍스트를 줄바꿈까지 그대로 전사해줘. "
    "읽을 수 없는 글자는 [?]로 표시하고, 임의로 고치거나 요약하지 마.",
)
print("[좋은 예]\n", good_ocr)

## Reasoning Prompt
단순 인식이 아니라 **근거를 먼저 설명한 뒤 답을 제시**하도록 요청하면
할루시네이션이 줄어듭니다. "모르면 모른다"고 답하게 만드는 것도 중요합니다.

In [ ]:
show(BAR_CHART)

In [ ]:
reasoning = ask(
    BAR_CHART,
    "이 막대그래프에서 값이 가장 큰 달과 가장 작은 달의 차이를 계산해줘. "
    "판단 근거(그래프의 어느 막대를 비교했는지)를 먼저 설명한 뒤 답을 제시하고, "
    "그래프에 없는 값은 추정하지 말고 '확인 불가'라고 답해.",
)
print(reasoning)

## Structured Output
후속 처리가 가능하도록 **JSON 스키마를 예시와 함께 명시**하고,
`ollama`의 `format="json"` 옵션(`ask_json`)으로 파싱 가능한 응답을 강제합니다.

In [ ]:
schema_prompt = '''
아래 JSON 스키마에 맞춰서만 답해. 다른 설명 문장은 출력하지 마.
{
  "category": "string",
  "color": "string",
  "pattern": "string",
  "size": "string"
}
'''
result = ask_json(PRODUCT, schema_prompt)
print(type(result), result)
assert set(result.keys()) >= {"category", "color", "pattern", "size"}

### 정리
- Caption → 범위/관점/용도 지정
- OCR → "전사" 지시 + 구조 유지 요청
- Reasoning → 근거 요구 + "모르면 모른다"
- Structured Output → 스키마 명시 + `format="json"` + 사후 검증(`assert`, `json.loads`)